In [1]:
import open3d as o3d
import rospy
import rosbag
import numpy as np

rosbag_path = '/home/seungwoo/airsim_pcl_logging.bag'
bag = rosbag.Bag(rosbag_path)


pcl_list = []
for topic, msg, t in bag.read_messages(topics=['/airsim/lopcl']):
    if msg._type == 'sensor_msgs/PointCloud2':
        pcl_msg = msg
        pcl = o3d.geometry.PointCloud()
        pcl.points = o3d.utility.Vector3dVector(np.frombuffer(pcl_msg.data, dtype=np.float32).reshape(-1, 3))
        pcl_list.append(pcl)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


INFO - 2025-09-26 05:04:33,960 - topics - topicmanager initialized


In [1]:
%pip install -q nest_asyncio
import nest_asyncio
nest_asyncio.apply()
import cosysairsim as airsim
client = airsim.MultirotorClient()
client.confirmConnection()  # 이제 RuntimeError 없어야 정상

import time
import numpy as np

# client.simSetVehiclePose(airsim.Pose(airsim.Vector3r(0,0,-2), airsim.to_quaternion(0,0,0)), True)
cur_time = time.time()
prev_time = None
change_count = 0
# measure update frequency of lidar
for i in range(200):
    data = client.getLidarData("lidar1")
    if prev_time is None:
        prev_time = data.time_stamp
    else:
        if data.time_stamp != prev_time:
            print("Lidar time difference: ", (data.time_stamp - prev_time) * 1e-9)
            change_count += 1
            prev_time = data.time_stamp

print("Total changes: ", change_count, "in 200 iterations", "HZ :", change_count / (time.time() - cur_time))


Note: you may need to restart the kernel to use updated packages.
Connected!
Client Ver:4 (Min Req: 4), Server Ver:4 (Min Req: 4)

Lidar time difference:  0.024231000000000003
Lidar time difference:  0.011547
Lidar time difference:  0.014309
Lidar time difference:  0.012046000000000001
Lidar time difference:  0.010914
Lidar time difference:  0.011765000000000001
Lidar time difference:  0.014881
Lidar time difference:  0.010893
Lidar time difference:  0.010315000000000001
Lidar time difference:  0.010536
Lidar time difference:  0.010561000000000001
Total changes:  11 in 200 iterations HZ : 78.96822603944874


In [ ]:
mesh = client.simGetMeshPositionVertexBuffers()

TransportError: Retry connection over the limit

: 

In [ ]:
# get cur airsim pose
pose = client.simGetVehiclePose()
print(pose)
client.simPause(False)
time.sleep(0.1)
client.simSetVehiclePose(airsim.Pose(airsim.Vector3r(0,0,0), airsim.euler_to_quaternion(0,0,0)), True)
# client.simPause(True)

<Pose> {   'orientation': <Quaternionr> {   'w_val': 1.0,
    'x_val': -0.0,
    'y_val': 0.0,
    'z_val': 0.0},
    'position': <Vector3r> {   'x_val': 5.76540315488927e-10,
    'y_val': -31.0,
    'z_val': -0.053663868457078934}}


In [40]:

for i in range(1000):
    cur_time = time.time() % np.pi
    client.simSetVehiclePose(airsim.Pose(airsim.Vector3r(1,1,0), airsim.euler_to_quaternion(0,0,cur_time)), True)

# client.simSetVehiclePose(airsim.Pose(airsim.Vector3r(0,0,-0.5), airsim.euler_to_quaternion(0,0,0)), True)


In [39]:
airsim.euler_to_quaternion(np.pi/2,0,0)

<Quaternionr> {   'w_val': 0.7071067811865476,
    'x_val': 0.7071067811865475,
    'y_val': 0.0,
    'z_val': 0.0}

In [54]:
data = client.getLidarData("lidar1")
pcl = o3d.geometry.PointCloud()
pcl.points = o3d.utility.Vector3dVector(np.array(data.point_cloud).reshape(-1, 3))

o3d.visualization.draw_geometries([pcl])

In [ ]:
# get pcl at pose grid

grid_x = np.linspace(-16, 0, 3)
grid_y = np.linspace(-31, 0, 3)



pcl_list = []

for x in grid_x:
    for y in grid_y:
        print(x,y)
        client.simSetVehiclePose(airsim.Pose(airsim.Vector3r(x,-y,-1), airsim.euler_to_quaternion(0,0,0)), True)
        client.simPause(False)
        client.simPause(True)

        data = client.getLidarData("lidar1")
        pcl = o3d.geometry.PointCloud()
        pcl.points = o3d.utility.Vector3dVector(np.array(data.point_cloud).reshape(-1, 3))
        pcl.translate(np.array([x,y,1]))

        pcl_list.append(pcl)
        

-16.0 -31.0
-16.0 -15.5
-16.0 0.0
-8.0 -31.0
-8.0 -15.5
-8.0 0.0
0.0 -31.0
0.0 -15.5
0.0 0.0


In [116]:
x = -10
y = -10
client.simSetVehiclePose(airsim.Pose(airsim.Vector3r(x,-y,-1), airsim.euler_to_quaternion(0,0,0)), True)
client.simPause(False)
time.sleep(0.01)
client.simPause(True)
data = client.getLidarData("lidar1")
pcl = o3d.geometry.PointCloud()
pcl.points = o3d.utility.Vector3dVector(np.array(data.point_cloud).reshape(-1, 3))
pcl.translate(np.array([x,y,1]))

pcl_list.append(pcl)

In [ ]:

o3d.visualization.draw_geometries([pcl])

: 